# GridLock 2.0 — Reproduce Final Submission

**Leaderboard submission:** `outputs/agent_b6_hwy_model_up.csv`  
**Public score:** 87.90 (exact: 87.89558)

This notebook generates the final submission end-to-end from the competition
`train.csv` and `test.csv` only. No external data is used.

**How to run:** place `train.csv` and `test.csv` in `dataset/`, install
`requirements.txt`, then **Run All** cells (~3–5 min on CPU).

Full methodology: see `docs/APPROACH.md`.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_config, set_seed
from src.data import load_raw
from src.model import train_model
from src.corrector import train_corrector
from experiments.gen_batch2 import build_raw_cell
from experiments.gen_batch4 import train_corr2

set_seed(42)
cfg = load_config()
print("Project root:", ROOT)
print("Seed:", cfg["seed"])

## 2. Load data

In [ ]:
train, test = load_raw(cfg)
print(f"train: {train.shape}  test: {test.shape}")
print(f"days in train: {sorted(train['day'].unique())}")
print(f"test slots: {test['slot'].min()}–{test['slot'].max()}")

## 3. Build four predictors

1. **raw_cell** — day-48 `(geohash, slot)` lookup with hierarchical fallback  
2. **model** — LightGBM on smoothed target encodings  
3. **corr** — day-over-day corrector (day-48 base + day-49 row features)  
4. **corr2** — corr + lat/lon + CatBoost/LightGBM ensemble

In [ ]:
print("Training predictors (this takes a few minutes)...")
raw_p   = build_raw_cell(train)(test)
model_p = train_model(train, transform="none", use_oof=True)[0](test)
corr_p  = train_corrector(train)[0](test)
corr2_p = train_corr2(train)(test)
print("Done.")
for name, arr in [("raw", raw_p), ("model", model_p), ("corr", corr_p), ("corr2", corr2_p)]:
    print(f"  {name:6s} mean={arr.mean():.4f}")

## 4. Per-RoadType four-way blend (champion recipe)

Weights = raw / model / corr / corr2:
- **Highway:** 0.45 / 0.25 / 0.15 / 0.15
- **Residential:** 0.65 / 0.10 / 0.12 / 0.13
- **Street & default:** 0.55 / 0.15 / 0.15 / 0.15

In [ ]:
CHAMPION = np.array([0.55, 0.15, 0.15, 0.15], dtype=float)

def weights_hwy_model_up(df: pd.DataFrame) -> np.ndarray:
    W = np.tile(CHAMPION, (len(df), 1))
    road = df["RoadType"].fillna("missing").to_numpy()
    W[road == "Highway"] = [0.45, 0.25, 0.15, 0.15]
    W[road == "Residential"] = [0.65, 0.10, 0.12, 0.13]
    return W

W = weights_hwy_model_up(test)
P = np.column_stack([raw_p, model_p, corr_p, corr2_p])
preds = np.clip((P * W).sum(axis=1), 0, 1)
print(f"blend mean={preds.mean():.4f}  min={preds.min():.4f}  max={preds.max():.4f}")

## 5. Write submission CSV

In [ ]:
out_dir = cfg["paths"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "agent_b6_hwy_model_up.csv")

sub = pd.DataFrame({"Index": test["Index"].astype(int), "demand": preds})
assert sub.shape == (41778, 2)
assert sub["Index"].tolist() == test["Index"].tolist()
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
sub.head()

## 6. Verification

In [ ]:
ref_path = Path(out_dir) / "agent_b6_hwy_model_up.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    diff = np.max(np.abs(ref["demand"].values - preds))
    print(f"rows: {len(sub)}  max_abs_diff vs on-disk file: {diff}")
else:
    print(f"rows: {len(sub)}  (no prior file to compare)")